# AMEFAL LOVEDA dataset

In [4]:
# import torch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import os
import gc
import cv2
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend — required for headless long runs
import matplotlib.pyplot as plt
import random
import shutil
from collections import deque
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import Dinov2Model, AutoImageProcessor
from sklearn.metrics import jaccard_score


# LoRA rank fixed at 4 — stable warm-start across all iterations
LORA_RANK  = 4
LORA_ALPHA = 4.0

# =============================================================================
#  GLOBAL SEED — set once here, applied everywhere
#  Change this single value if you want a different but still reproducible run.
# =============================================================================
GLOBAL_SEED = 42


def set_global_seed(seed: int = GLOBAL_SEED) -> None:
    """
    
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)          # covers multi-GPU
    # Force deterministic CUDA kernels (slight speed cost, full reproducibility)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED']       = str(seed)
    print(f"✓ Global seed set to {seed} — fully reproducible run")


# =============================================================================
#  BLOCK 1: DINOv2 Fine-tuning
# =============================================================================

def select_dinov2_config():
    print("\n" + "=" * 70)
    print("DINOV2 FINE-TUNING METHOD SELECTION")
    print("=" * 70)

    while True:
        t = input("  Enter 1 or 2: ").strip()
        if t == '1':
            training_type = 'supervised';      print("  ✓ Supervised");      break
        if t == '2':
            training_type = 'self_supervised'; print("  ✓ Self-Supervised"); break
        print("  ⚠ Please enter 1 or 2.")

    while True:
        m = input("  Enter 1 or 2: ").strip()
        if m == '1':
            method = 'multilabel'; print("  ✓ Multi-Label"); break
        if m == '2':
            method = 'lora';       print("  ✓ LoRA");        break
        print("  ⚠ Please enter 1 or 2.")

    config = {'training_type': training_type, 'method': method}
    print(f"\n✓ DINOv2 config selected:")
    print(f"  Training : {training_type.replace('_',' ').title()}")
    print(f"  Method   : {method.title()}")

    confirm = input("\nConfirm? (y/n): ").strip().lower()
    if confirm != 'y':
        print("Let's try again...\n")
        return select_dinov2_config()
    return config


def load_geodinov2_encoder(device, model_name='facebook/dinov2-base', cache_dir=None):
    print(f"Loading {model_name}...")
    try:
        model     = Dinov2Model.from_pretrained(model_name, cache_dir=cache_dir)
        processor = AutoImageProcessor.from_pretrained(model_name, cache_dir=cache_dir)
        model.to(device)
        model.eval()
        for param in model.parameters():
            param.requires_grad = False
        if 'small' in model_name:   embed_dim = 384
        elif 'large' in model_name: embed_dim = 1024
        elif 'giant' in model_name: embed_dim = 1536
        else:                       embed_dim = 768
        print(f"✓ DINOv2 loaded | dim={embed_dim} | device={device}")
        return model, processor, embed_dim
    except Exception as e:
        print(f"❌ Failed to load DINOv2: {e}")
        raise


def get_geodinov2_embeddings(images, geodino_model, geodino_processor):
    B, C, H, W = images.shape
    device      = images.device
    geodino_model = geodino_model.to(device)
    images_pil = []
    for i in range(B):
        img_np = (images[i].cpu().permute(1,2,0).numpy() * 255).astype('uint8')
        images_pil.append(Image.fromarray(img_np))
    inputs = geodino_processor(images=images_pil, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = geodino_model(**inputs)
        if hasattr(outputs, 'last_hidden_state'):
            embeddings = outputs.last_hidden_state[:, 0, :]
        elif hasattr(outputs, 'pooler_output'):
            embeddings = outputs.pooler_output
        else:
            raise ValueError("Unexpected model output format")
        embeddings = F.normalize(embeddings, dim=-1)
    return embeddings


def _tensors_to_pil(t):
    return [Image.fromarray(
                (t[i].cpu().permute(1,2,0).numpy()*255).astype('uint8'))
            for i in range(t.shape[0])]

def _get_embed_dim(model_name):
    if 'small' in model_name: return 384
    if 'large' in model_name: return 1024
    if 'giant' in model_name: return 1536
    return 768

def _build_multilabel_targets(masks, num_classes=8, min_proportion=0.05, device='cpu'):
    B     = masks.shape[0]
    total = masks.shape[1] * masks.shape[2]
    tgt   = torch.zeros(B, num_classes, dtype=torch.float32)
    for i in range(B):
        flat          = masks[i].view(-1).cpu().numpy()
        classes, cnts = np.unique(flat, return_counts=True)
        for c, n in zip(classes, cnts):
            if 0 <= c < num_classes and n / total >= min_proportion:
                tgt[i, c] = n / total
        s = tgt[i].sum()
        if s > 0:
            tgt[i] /= s
    return tgt.to(device)


class LoRALinear(nn.Module):
    def __init__(self, original_linear, rank=4, alpha=4.0):
        super().__init__()
        d_in  = original_linear.in_features
        d_out = original_linear.out_features
        self.register_buffer('weight', original_linear.weight.data.clone())
        if original_linear.bias is not None:
            self.register_buffer('bias', original_linear.bias.data.clone())
        else:
            self.register_buffer('bias', None)
        self.lora_A = nn.Parameter(torch.empty(rank, d_in))
        self.lora_B = nn.Parameter(torch.zeros(d_out, rank))
        self.scale  = alpha / rank
        nn.init.normal_(self.lora_A, std=0.02)

    def forward(self, x):
        return (F.linear(x, self.weight, self.bias) +
                F.linear(F.linear(x, self.lora_A), self.lora_B) * self.scale)


def _apply_lora(dino_model, rank=4, alpha=4.0, layers=(10, 11)):
    replaced = 0
    for idx in layers:
        try:
            attn = dino_model.encoder.layer[idx].attention.attention
            attn.query = LoRALinear(attn.query, rank, alpha)
            attn.key   = LoRALinear(attn.key,   rank, alpha)
            attn.value = LoRALinear(attn.value, rank, alpha)
            replaced  += 3
        except AttributeError as e:
            print(f"  ⚠ LoRA injection failed at layer {idx}: {e}")
    trainable = sum(p.numel() for p in dino_model.parameters() if p.requires_grad)
    print(f"  LoRA: {replaced} projections injected | trainable={trainable:,} params")
    return dino_model


def _load_lora_weights_into_model(dino_model, lora_weights_dict, device):
    if lora_weights_dict is None:
        print("  ℹ No previous LoRA weights — starting from random init")
        return dino_model
    loaded, skipped = 0, 0
    model_state = dino_model.state_dict()
    for k, v in lora_weights_dict.items():
        if k in model_state and model_state[k].shape == v.shape:
            model_state[k] = v.to(device)
            loaded += 1
        else:
            skipped += 1
    dino_model.load_state_dict(model_state, strict=False)
    print(f"  ✓ LoRA warm-start: loaded={loaded} keys, skipped={skipped} keys")
    if skipped > 0 and loaded == 0:
        print(f"  ⚠ WARNING: ALL {skipped} LoRA keys skipped — shapes mismatched!")
    elif skipped > 0:
        print(f"  ⚠ {skipped} keys skipped (shape mismatch) — partial warm-start.")
    return dino_model


def _finetune_supervised_multilabel(train_loader, device, num_classes,
                                     num_epochs, lr, model_name,
                                     cache_dir=None, existing_lora_weights=None):
    print(f"\n{'='*70}")
    print("DINOV2 FINE-TUNE: SUPERVISED + MULTI-LABEL")
    print(f"  Samples: {len(train_loader.dataset)} | Epochs: {num_epochs}")
    print(f"{'='*70}")

    dino = Dinov2Model.from_pretrained(model_name, cache_dir=cache_dir)
    proc = AutoImageProcessor.from_pretrained(model_name, cache_dir=cache_dir)
    edim = _get_embed_dim(model_name)
    for p in dino.parameters(): p.requires_grad = False

    head = nn.Sequential(
        nn.Linear(edim, 512), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(512, 256),  nn.ReLU(), nn.Dropout(0.1),
        nn.Linear(256, num_classes)
    ).to(device)
    bce = nn.BCEWithLogitsLoss()

    print("\nStage 1/2 — head only (backbone frozen)")
    print("-" * 70)
    opt1 = optim.AdamW(head.parameters(), lr=lr * 10)
    for ep in range(5):
        head.train(); loss_sum = 0; nb = 0
        for imgs, masks in train_loader:
            imgs  = imgs.to(device); masks = masks.to(device)
            tgt   = _build_multilabel_targets(masks, num_classes, device=device)
            inp   = proc(images=_tensors_to_pil(imgs), return_tensors="pt")
            inp   = {k: v.to(device) for k, v in inp.items()}
            with torch.no_grad():
                cls = dino(**inp).last_hidden_state[:, 0, :]
            opt1.zero_grad()
            loss = bce(head(cls), tgt); loss.backward(); opt1.step()
            loss_sum += loss.item(); nb += 1
        print(f"  Epoch {ep+1}/5 | Loss: {loss_sum/nb:.4f}")
    print(f"✓ Stage 1 complete!\n")

    print("Stage 2/2 — layers 10-11 unfrozen")
    print("-" * 70)
    for name, p in dino.named_parameters():
        if 'encoder.layer.10' in name or 'encoder.layer.11' in name:
            p.requires_grad = True
    trainable = sum(p.numel() for p in dino.parameters() if p.requires_grad)
    print(f"  Trainable params: {trainable:,}")
    dino = dino.to(device)

    opt2 = optim.AdamW([
        {'params': [p for p in dino.parameters() if p.requires_grad], 'lr': lr/10},
        {'params': head.parameters(), 'lr': lr}
    ])
    best, patience = float('inf'), 0
    for ep in range(num_epochs):
        dino.train(); head.train(); loss_sum = 0; nb = 0
        for imgs, masks in train_loader:
            imgs  = imgs.to(device); masks = masks.to(device)
            tgt   = _build_multilabel_targets(masks, num_classes, device=device)
            inp   = proc(images=_tensors_to_pil(imgs), return_tensors="pt")
            inp   = {k: v.to(device) for k, v in inp.items()}
            opt2.zero_grad()
            cls  = F.normalize(dino(**inp).last_hidden_state[:,0,:], dim=-1)
            loss = bce(head(cls), tgt); loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(dino.parameters()) + list(head.parameters()), 1.0)
            opt2.step()
            loss_sum += loss.item(); nb += 1
        avg = loss_sum / max(1, nb)
        if (ep+1) % 5 == 0:
            print(f"  Epoch {ep+1}/{num_epochs} | Loss: {avg:.4f}")
        if avg < best: best = avg; patience = 0
        else:
            patience += 1
            if patience >= 10: print(f"  ⚠ Early stopping at epoch {ep+1}"); break

    print(f"\n✓ Stage 2 complete! | Best loss: {best:.4f}")
    print(f"{'='*70}\n")
    for p in dino.parameters(): p.requires_grad = False
    dino.eval()
    return dino, proc


def _finetune_supervised_lora(train_loader, device, num_classes,
                               num_epochs, lr, model_name,
                               cache_dir=None, existing_lora_weights=None,
                               use_gradient_checkpointing=False):
    num_samples    = len(train_loader.dataset)
    lora_rank, lora_alpha = LORA_RANK, LORA_ALPHA

    print(f"\n{'='*70}")
    print("DINOV2 FINE-TUNE: SUPERVISED + LORA")
    print(f"  Samples : {num_samples}")
    print(f"  Rank    : {lora_rank} (fixed)  |  Alpha: {lora_alpha:.0f} (fixed)  |  Scale: {lora_alpha/lora_rank:.1f}")
    print(f"  Epochs  : {num_epochs}")
    print(f"  Warm-start LoRA: {'YES' if existing_lora_weights else 'NO (first time)'}")
    print(f"{'='*70}\n")

    dino = Dinov2Model.from_pretrained(model_name, cache_dir=cache_dir)
    proc = AutoImageProcessor.from_pretrained(model_name, cache_dir=cache_dir)
    edim = _get_embed_dim(model_name)
    for p in dino.parameters(): p.requires_grad = False

    dino = _apply_lora(dino, rank=lora_rank, alpha=lora_alpha)
    dino = dino.to(device)

    if existing_lora_weights is not None:
        dino = _load_lora_weights_into_model(dino, existing_lora_weights, device)
    else:
        print("  ℹ First fine-tune — LoRA starts from random init (lora_B=0)")

    if use_gradient_checkpointing:
        try:
            dino.gradient_checkpointing_enable()
            print("  ✓ Gradient checkpointing ENABLED (saves ~30% memory)")
        except Exception as e:
            print(f"  ⚠ Could not enable gradient checkpointing: {e}")

    head = nn.Sequential(
        nn.Linear(edim, 512), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(512, 256),  nn.ReLU(), nn.Dropout(0.1),
        nn.Linear(256, num_classes)
    ).to(device)
    bce         = nn.BCEWithLogitsLoss()
    lora_params = [p for p in dino.parameters() if p.requires_grad]

    print(f"  Total trainable LoRA params: {sum(p.numel() for p in lora_params):,}")
    print(f"  Head params:                 {sum(p.numel() for p in head.parameters()):,}")
    print("Training...")
    print("-" * 70)

    opt   = optim.AdamW([{'params': lora_params,       'lr': lr},
                          {'params': head.parameters(), 'lr': lr}],
                         weight_decay=0.01)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=num_epochs, eta_min=lr/10)
    best, patience = float('inf'), 0

    for ep in range(num_epochs):
        dino.train(); head.train(); loss_sum = 0; nb = 0
        for imgs, masks in train_loader:
            imgs  = imgs.to(device); masks = masks.to(device)
            tgt   = _build_multilabel_targets(masks, num_classes, device=device)
            inp   = proc(images=_tensors_to_pil(imgs), return_tensors="pt")
            inp   = {k: v.to(device) for k, v in inp.items()}
            opt.zero_grad()
            cls  = F.normalize(dino(**inp).last_hidden_state[:,0,:], dim=-1)
            loss = bce(head(cls), tgt); loss.backward()
            torch.nn.utils.clip_grad_norm_(lora_params + list(head.parameters()), 1.0)
            opt.step()
            loss_sum += loss.item(); nb += 1
        sched.step()
        avg = loss_sum / max(1, nb)
        if (ep+1) % 3 == 0 or ep == 0:
            print(f"  Epoch {ep+1:3d}/{num_epochs} | Loss: {avg:.4f} | "
                  f"LR: {sched.get_last_lr()[0]:.2e}")
        if avg < best: best = avg; patience = 0
        else:
            patience += 1
            if patience >= 10: print(f"  ⚠ Early stopping at epoch {ep+1}"); break

    print(f"\n✓ Done! | Best loss: {best:.4f}")
    print(f"{'='*70}\n")
    dino.eval()
    for p in dino.parameters(): p.requires_grad = False
    return dino, proc


class _PatchReconHead(nn.Module):
    def __init__(self, embed_dim=768, patch_size=14):
        super().__init__()
        ppx = patch_size * patch_size * 3
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim * 2),
            nn.GELU(),
            nn.Linear(embed_dim * 2, ppx)
        )
    def forward(self, x): return self.head(x)


def _extract_patches(images, patch_size=14):
    B, C, H, W = images.shape
    patches = []
    for ph in range(H // patch_size):
        for pw in range(W // patch_size):
            p = images[:, :, ph*patch_size:(ph+1)*patch_size,
                              pw*patch_size:(pw+1)*patch_size]
            patches.append(p.reshape(B, -1))
    return torch.stack(patches, dim=1)


def _finetune_self_supervised(image_dirs, device, method, num_epochs, lr, model_name,
                               cache_dir=None, existing_lora_weights=None,
                               mask_ratio=0.75, batch_size=8):
    class _ImgDataset(Dataset):
        def __init__(self, dirs):
            self.paths = []
            for d in dirs:
                if os.path.isdir(d):
                    for f in os.listdir(d):
                        if f.lower().endswith(('.jpg','.jpeg','.png','.tif','.tiff')):
                            self.paths.append(os.path.join(d, f))
            print(f"  Self-supervised dataset: {len(self.paths)} images")
        def __len__(self): return len(self.paths)
        def __getitem__(self, idx):
            img = cv2.imread(self.paths[idx])
            if img is None:
                img = np.zeros((224,224,3), dtype=np.uint8)
            else:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)   # BGR → RGB
            img = cv2.resize(img, (224, 224))
            return torch.tensor(img, dtype=torch.float32).permute(2,0,1) / 255.0

    lora_rank, lora_alpha = LORA_RANK, LORA_ALPHA

    print(f"\n{'='*70}")
    print(f"DINOV2 FINE-TUNE: SELF-SUPERVISED + {method.upper()}")
    print(f"  Dirs  : {image_dirs}")
    print(f"  Mask  : {mask_ratio*100:.0f}%  |  Epochs: {num_epochs}")
    print(f"  Rank  : {lora_rank} (fixed)  |  Alpha: {lora_alpha:.0f} (fixed)  |  Scale: {lora_alpha/lora_rank:.1f}")
    print(f"{'='*70}\n")

    dino = Dinov2Model.from_pretrained(model_name, cache_dir=cache_dir)
    proc = AutoImageProcessor.from_pretrained(model_name, cache_dir=cache_dir)
    edim = _get_embed_dim(model_name)
    for p in dino.parameters(): p.requires_grad = False

    if method == 'lora':
        dino = _apply_lora(dino, rank=lora_rank, alpha=lora_alpha)
        dino = dino.to(device)
        if existing_lora_weights is not None:
            dino = _load_lora_weights_into_model(dino, existing_lora_weights, device)
    else:
        dino = dino.to(device)
        for name, p in dino.named_parameters():
            if 'encoder.layer.10' in name or 'encoder.layer.11' in name:
                p.requires_grad = True
        trainable = sum(p.numel() for p in dino.parameters() if p.requires_grad)
        print(f"  Unfrozen params: {trainable:,}")

    recon_head = _PatchReconHead(edim).to(device)
    dataset    = _ImgDataset(image_dirs)
    if len(dataset) == 0:
        print("  ⚠ No images found — skipping self-supervised fine-tuning")
        return dino, proc

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0,
                        pin_memory=torch.cuda.is_available())

    backbone_params = [p for p in dino.parameters() if p.requires_grad]
    opt   = optim.AdamW([{'params': backbone_params,          'lr': lr/10},
                          {'params': recon_head.parameters(), 'lr': lr}], weight_decay=0.01)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=num_epochs, eta_min=lr/100)
    mse   = nn.MSELoss()
    n_patches = 16 * 16  # 224px / 14px = 16 grid → 256 patches

    best, patience = float('inf'), 0
    for ep in range(num_epochs):
        dino.train(); recon_head.train(); loss_sum = 0; nb = 0
        for imgs in loader:
            imgs = imgs.to(device)
            patches = _extract_patches(imgs)
            num_mask = int(n_patches * mask_ratio)
            mask_idx = torch.randperm(n_patches)[:num_mask]
            inp   = proc(images=_tensors_to_pil(imgs), return_tensors="pt")
            inp   = {k: v.to(device) for k, v in inp.items()}
            opt.zero_grad()
            hidden = dino(**inp).last_hidden_state[:, 1:, :]  # skip CLS
            masked_hidden = hidden[:, mask_idx, :]
            recon  = recon_head(masked_hidden)
            target = patches[:, mask_idx, :]
            loss   = mse(recon, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(backbone_params + list(recon_head.parameters()), 1.0)
            opt.step()
            loss_sum += loss.item(); nb += 1
        sched.step()
        avg = loss_sum / max(1, nb)
        if (ep+1) % 3 == 0 or ep == 0:
            print(f"  Epoch {ep+1:3d}/{num_epochs} | Recon Loss: {avg:.4f} | "
                  f"LR: {sched.get_last_lr()[0]:.2e}")
        if avg < best: best = avg; patience = 0
        else:
            patience += 1
            if patience >= 10: print(f"  ⚠ Early stopping at epoch {ep+1}"); break

    print(f"\n✓ Done! | Best recon loss: {best:.4f}")
    print(f"{'='*70}\n")
    dino.eval()
    for p in dino.parameters(): p.requires_grad = False
    return dino, proc


def run_dinov2_finetuning(config, train_loader, device, num_classes,
                           num_epochs, lr, model_name,
                           train_image_dir=None, pool_image_dir=None,
                           batch_size=4, cache_dir=None,
                           existing_dino_model=None,
                           use_gradient_checkpointing=False):
    training_type = config['training_type']
    method        = config['method']

    existing_lora_weights = None
    if existing_dino_model is not None and method == 'lora':
        existing_lora_weights = {
            k: v.detach().clone()
            for k, v in existing_dino_model.state_dict().items()
            if 'lora_A' in k or 'lora_B' in k
        }
        if existing_lora_weights:
            print(f"  ✓ Extracted {len(existing_lora_weights)} LoRA weight tensors for warm-start")
        else:
            print("  ℹ No LoRA keys found in existing model")
            existing_lora_weights = None

    if training_type == 'supervised':
        if method == 'multilabel':
            return _finetune_supervised_multilabel(
                train_loader, device, num_classes, num_epochs, lr, model_name,
                cache_dir=cache_dir, existing_lora_weights=existing_lora_weights)
        else:
            return _finetune_supervised_lora(
                train_loader, device, num_classes, num_epochs, lr, model_name,
                cache_dir=cache_dir, existing_lora_weights=existing_lora_weights,
                use_gradient_checkpointing=use_gradient_checkpointing)
    else:
        image_dirs = []
        if train_image_dir and os.path.isdir(train_image_dir):
            image_dirs.append(train_image_dir)
        if pool_image_dir and os.path.isdir(pool_image_dir):
            image_dirs.append(pool_image_dir)
        return _finetune_self_supervised(
            image_dirs, device, method, num_epochs, lr, model_name,
            cache_dir=cache_dir, existing_lora_weights=existing_lora_weights,
            batch_size=batch_size)


def save_finetuned_dinov2(dino_model, dino_processor, save_path):
    state_dict = {k: v for k, v in dino_model.state_dict().items()
                  if ('encoder.layer.11' in k or 'encoder.layer.10' in k
                      or 'lora_A' in k or 'lora_B' in k)}
    lora_keys  = [k for k in state_dict if 'lora_A' in k or 'lora_B' in k]
    torch.save({
        'finetuned_layers': state_dict,
        'model_type': 'dinov2_finetuned',
        'num_lora_keys': len(lora_keys),
    }, save_path)
    print(f"✓ Fine-tuned DINOv2 saved: {save_path}")
    print(f"  Saved {len(state_dict)} weight tensors ({len(lora_keys)} LoRA keys)")


def load_finetuned_dinov2(dino_model, load_path, device):
    if os.path.exists(load_path):
        print(f"Loading fine-tuned DINOv2 from: {load_path}")
        checkpoint = torch.load(load_path, map_location=device, weights_only=False)
        dino_model.load_state_dict(checkpoint['finetuned_layers'], strict=False)
        n_lora = checkpoint.get('num_lora_keys', '?')
        print(f"✓ Fine-tuned DINOv2 loaded! (LoRA keys: {n_lora})")
        return True
    return False


# =============================================================================
#  BLOCK 2: UNet Segmentation Model
# =============================================================================

class UNetWithLatentSpace(nn.Module):
    def __init__(self, num_classes=8, embedding_dim=256):
        super(UNetWithLatentSpace, self).__init__()
        self.num_classes   = num_classes
        self.embedding_dim = embedding_dim

        self.encoder_block1 = self.conv_block(3, 64)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.encoder_block2 = self.conv_block(64, 128)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.encoder_block3 = self.conv_block(128, 256)
        self.pool3 = nn.MaxPool2d(2, 2)
        self.encoder_block4 = self.conv_block(256, 512)
        self.pool4 = nn.MaxPool2d(2, 2)
        self.bottleneck = self.conv_block(512, 1024)
        self.embedding_projection = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(1024, 512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Linear(256, embedding_dim)
        )
       
        self.upconv1 = nn.ConvTranspose2d(1024, 512, 3, 2, 1, 1)
        self.decoder_block1 = self.conv_block(1024, 512)
        self.upconv2 = nn.ConvTranspose2d(512, 256, 3, 2, 1, 1)
        self.decoder_block2 = self.conv_block(512, 256)
        self.upconv3 = nn.ConvTranspose2d(256, 128, 3, 2, 1, 1)
        self.decoder_block3 = self.conv_block(256, 128)
        self.upconv4 = nn.ConvTranspose2d(128, 64, 3, 2, 1, 1)
        self.decoder_block4 = self.conv_block(128, 64)
        self.output_conv = nn.Conv2d(64, num_classes, 3, 1, 1)

    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(out_channels),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x, return_embedding=False, return_both=False):
        enc1 = self.encoder_block1(x);     enc1_pooled = self.pool1(enc1)
        enc2 = self.encoder_block2(enc1_pooled); enc2_pooled = self.pool2(enc2)
        enc3 = self.encoder_block3(enc2_pooled); enc3_pooled = self.pool3(enc3)
        enc4 = self.encoder_block4(enc3_pooled); enc4_pooled = self.pool4(enc4)
        bottleneck_features = self.bottleneck(enc4_pooled)
        embedding = self.embedding_projection(bottleneck_features)

        if return_embedding:
            return embedding

        dec1 = self.upconv1(bottleneck_features)
        dec1 = torch.cat([dec1, enc4], dim=1); dec1 = self.decoder_block1(dec1)
        dec2 = self.upconv2(dec1)
        dec2 = torch.cat([dec2, enc3], dim=1); dec2 = self.decoder_block2(dec2)
        dec3 = self.upconv3(dec2)
        dec3 = torch.cat([dec3, enc2], dim=1); dec3 = self.decoder_block3(dec3)
        dec4 = self.upconv4(dec3)
        dec4 = torch.cat([dec4, enc1], dim=1); dec4 = self.decoder_block4(dec4)
        segmentation_output = self.output_conv(dec4)

        if return_both:
            return segmentation_output, embedding
        return segmentation_output


def train_model_with_warm_start(train_loader, model, criterion, optimizer,
                                num_epochs, device, use_multi_gpu=False,
                                previous_model_path=None, val_loader=None,
                                early_stopping_patience=8, is_first_iteration=False):
    model = model.to(device)

    if previous_model_path and os.path.exists(previous_model_path):
        print(f"\n{'='*70}")
        print("WARM STARTING FROM PREVIOUS MODEL")
        print(f"{'='*70}")
        print(f"Loading weights from: {previous_model_path}")
        checkpoint = torch.load(previous_model_path, map_location=device, weights_only=False)
        if isinstance(model, nn.DataParallel):
            model.module.load_state_dict(checkpoint['model_state_dict'])
        else:
            model.load_state_dict(checkpoint['model_state_dict'])
        print(f"✓ Previous model weights loaded successfully!")
        print(f"  Previous iteration: {checkpoint.get('iteration', 'unknown')}")
        print(f"{'='*70}\n")
    else:
        print(f"\n{'='*70}")
        print("TRAINING FROM SCRATCH (First Iteration)")
        print(f"{'='*70}\n")

    if use_multi_gpu and torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
        model = nn.DataParallel(model, device_ids=list(range(torch.cuda.device_count())))

    train_losses, val_losses, val_mious = [], [], []
    best_val_miou   = 0.0
    patience_counter = 0
    best_model_state = None
    use_early_stopping = (not is_first_iteration) and (val_loader is not None)

    if is_first_iteration:
        print(f"First iteration: Training for {num_epochs} epochs (NO early stopping)\n")
    elif use_early_stopping:
        print(f"Training with EARLY STOPPING (patience={early_stopping_patience}, max={num_epochs} epochs)\n")
    else:
        print(f"Training for {num_epochs} epochs (no validation)\n")

    for epoch in range(num_epochs):
        model.train()
        epoch_loss   = 0
        batch_count  = 0
        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            epoch_loss  += loss.item()
            batch_count += 1

        avg_train_loss = epoch_loss / max(1, batch_count)
        train_losses.append(avg_train_loss)

        if val_loader is not None:
            model.eval()
            val_loss = 0; val_batch_count = 0
            all_preds = []; all_targets = []
            with torch.no_grad():
                for batch in val_loader:
                    if len(batch) == 3: images, targets, _ = batch
                    else:               images, targets    = batch
                    images, targets = images.to(device), targets.to(device)
                    if isinstance(model, nn.DataParallel):
                        outputs = model.module(images)
                    else:
                        outputs = model(images)
                    loss = criterion(outputs, targets)
                    val_loss       += loss.item()
                    val_batch_count += 1
                    preds = torch.argmax(outputs, dim=1)
                    all_preds.extend(preds.cpu().numpy().flatten())
                    all_targets.extend(targets.cpu().numpy().flatten())

            avg_val_loss = val_loss / max(1, val_batch_count)
            val_losses.append(avg_val_loss)
            all_preds   = np.array(all_preds)
            all_targets = np.array(all_targets)
            iou_per_class = []
            n_cls = model.module.num_classes if isinstance(model, nn.DataParallel) else model.num_classes
            for cls in range(1, n_cls):  # skip class 0 (Ignore/NoData)
                if (all_targets == cls).sum() == 0: continue
                iou = jaccard_score(all_targets == cls, all_preds == cls, zero_division=0)
                iou_per_class.append(iou)
            val_miou = np.mean(iou_per_class) if iou_per_class else 0.0
            val_mious.append(val_miou)

            if val_miou > best_val_miou:
                best_val_miou    = val_miou
                patience_counter = 0
                _sd = (model.module.state_dict() if isinstance(model, nn.DataParallel)
                       else model.state_dict())
                best_model_state = {k: v.cpu().clone() for k, v in _sd.items()}
                improvement = " (improved)"
            else:
                patience_counter += 1
                improvement = f"(no improvement: {patience_counter}/{early_stopping_patience})"

            if (epoch + 1) % 5 == 0:
                print(f"Epoch {epoch+1:3d}/{num_epochs} | "
                      f"Train Loss: {avg_train_loss:.4f} | "
                      f"Val Loss: {avg_val_loss:.4f} | "
                      f"Val mIoU: {val_miou:.4f} {improvement}")

            if use_early_stopping and patience_counter >= early_stopping_patience:
                print(f"\n Early stopping triggered at epoch {epoch + 1}")
                print(f"  Best validation mIoU: {best_val_miou:.4f}")
                if best_model_state is not None:
                    if isinstance(model, nn.DataParallel):
                        model.module.load_state_dict(best_model_state)
                    else:
                        model.load_state_dict(best_model_state)
                    print(f"   Restored best model weights")
                break
        else:
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1:3d}/{num_epochs} | Train Loss: {avg_train_loss:.4f}")

    epochs_completed = len(train_losses)
    final_val_miou   = val_mious[-1] if val_mious else 0.0
    return model, train_losses, epochs_completed, val_losses, val_mious


def save_model(model, iteration_dir, iteration, num_classes=8):
    model_path  = os.path.join(iteration_dir, f'model_iteration_{iteration}.pth')
    model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    torch.save({
        'model_state_dict': model_state,
        'iteration':        iteration,
        'num_classes':      num_classes,
        'model_type':       'UNetWithLatentSpace'
    }, model_path)
    print(f" Model saved: {model_path}")
    return model_path


# =============================================================================
#  BLOCK 3: Metrics and Evaluation
# =============================================================================

def compute_miou(preds, targets, num_classes=8, ignore_index=0):
    iou_per_class = np.zeros(num_classes)
    preds   = preds.flatten()
    targets = targets.flatten()
    for cls in range(num_classes):
        if cls == ignore_index or (targets == cls).sum() == 0:
            iou_per_class[cls] = np.nan
            continue
        iou_per_class[cls] = jaccard_score(targets == cls, preds == cls, zero_division=0)
    return np.nanmean(iou_per_class), iou_per_class


def compute_class_accuracies(preds, targets, num_classes=8):
    class_accuracies = []
    for cls in range(num_classes):
        mask = (targets == cls)
        if mask.sum() > 0:
            class_accuracies.append(np.mean(preds[mask] == targets[mask]))
        else:
            class_accuracies.append(0.0)
    return class_accuracies


def validate_model(model, val_loader, criterion, device, num_classes=8):
    base_model = model.module if isinstance(model, nn.DataParallel) else model
    base_model.eval()
    total_correct = 0; total_pixels = 0; total_loss = 0
    all_preds = []; all_targets = []

    with torch.no_grad():
        for batch in val_loader:
            if len(batch) == 3: images, targets, _ = batch
            else:               images, targets    = batch
            images, targets = images.to(device), targets.to(device)
            outputs = base_model(images)
            loss    = criterion(outputs, targets)
            total_loss    += loss.item()
            preds          = torch.argmax(outputs, dim=1)
            total_correct += (preds == targets).sum().item()
            total_pixels  += targets.numel()
            all_preds.extend(preds.cpu().numpy().flatten())
            all_targets.extend(targets.cpu().numpy().flatten())

    accuracy  = total_correct / total_pixels if total_pixels > 0 else 0.0
    avg_loss  = total_loss / len(val_loader) if len(val_loader) > 0 else float('inf')
    all_preds   = np.array(all_preds)
    all_targets = np.array(all_targets)
    miou, class_ious     = compute_miou(all_preds, all_targets, num_classes=num_classes)
    class_accuracies     = compute_class_accuracies(all_preds, all_targets, num_classes=num_classes)
    return accuracy, avg_loss, miou, class_ious, class_accuracies


def compute_diversity_score(candidate_embedding, train_embeddings, metric='cosine'):
    if len(train_embeddings) == 0: return 1.0
    from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
    candidate_np = candidate_embedding.cpu().numpy().reshape(1, -1)
    train_np     = train_embeddings.cpu().numpy()
    if metric == 'cosine':
        similarities    = cosine_similarity(candidate_np, train_np)[0]
        diversity       = 1.0 - np.mean(similarities)
    elif metric == 'euclidean':
        distances       = euclidean_distances(candidate_np, train_np)[0]
        diversity       = np.mean(distances) / np.sqrt(2)
    else:
        raise ValueError(f"Unknown metric: {metric}")
    return diversity


# =============================================================================
#  BLOCK 4: MLP Meta-Learner — TinyWeightMLP + MLPMetaWeightLearner
#  Per-sample weight prediction trained on val mIoU improvement as reward.
# =============================================================================

class TinyWeightMLP(nn.Module):
    """Tiny MLP that predicts per-sample combination weights from method scores."""
    def __init__(self, selected_methods, hidden_dim=32):
        super(TinyWeightMLP, self).__init__()
        self.selected_methods = selected_methods
        input_dim  = len(selected_methods)
        output_dim = len(selected_methods)
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, output_dim),
            nn.Softmax(dim=-1)
        )
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight, gain=0.1)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0.0)

    def forward(self, x):
        if x.dim() == 1:
            return self.network(x.unsqueeze(0)).squeeze(0)
        return self.network(x)


class MLPMetaWeightLearner:
    """Meta-learner that trains TinyWeightMLP using validation mIoU as reward signal."""

    def __init__(self, selected_methods, hidden_dim=32, learning_rate=0.001, device='cuda'):
        self.device           = device
        self.selected_methods = selected_methods
        self.num_methods      = len(selected_methods)
        self.mlp              = TinyWeightMLP(selected_methods, hidden_dim).to(device)
        self.optimizer        = optim.Adam(self.mlp.parameters(), lr=learning_rate)
        self.scheduler        = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='max', factor=0.5, patience=2, verbose=False)
        self.experience_buffer = deque(maxlen=100)
        self.best_val_miou     = 0.0
        self.best_mlp_state    = None
        self.history           = []
        self.current_weights   = [1.0 / self.num_methods] * self.num_methods
        self.previous_val_miou = 0.0

    # ── inference ─────────────────────────────────────────────────────────
    def predict_sample_weights(self, scores_tensor):
        """scores_tensor: [N, num_methods] → weights: [N, num_methods]"""
        self.mlp.eval()
        with torch.no_grad():
            return self.mlp(scores_tensor)

    def compute_hybrid_scores_with_mlp(self, pool_scores, save_path=None):
        """Compute per-sample hybrid scores using MLP-predicted weights."""
        score_list = [torch.tensor(pool_scores[m], dtype=torch.float32, device=self.device)
                      for m in self.selected_methods]
        scores_tensor     = torch.stack(score_list, dim=1)           # [N, M]
        predicted_weights = self.predict_sample_weights(scores_tensor) # [N, M]
        hybrid_scores     = (predicted_weights * scores_tensor).sum(dim=1)

        weights_np = predicted_weights.cpu().numpy()
        scores_np  = scores_tensor.cpu().numpy()
        hybrid_np  = hybrid_scores.cpu().numpy()

        detailed_log = []
        for i in range(len(hybrid_np)):
            entry = {'sample_idx': i, 'hybrid_score': float(hybrid_np[i])}
            for j, m in enumerate(self.selected_methods):
                entry[m]             = float(scores_np[i, j])
                entry[f'weight_{m}'] = float(weights_np[i, j])
            detailed_log.append(entry)

        if save_path:
            pd.DataFrame(detailed_log).to_csv(save_path, index=False)
            print(f"   ✓ MLP predictions saved: {save_path}")

        return hybrid_np, weights_np, detailed_log

    # ── experience ────────────────────────────────────────────────────────
    def store_experience(self, pool_scores, selected_indices):
        """Store experience WITHOUT reward (filled in next iteration)."""
        self.experience_buffer.append({
            'pool_scores':      pool_scores,
            'selected_indices': selected_indices,
            'reward':           None
        })

    # ── training ──────────────────────────────────────────────────────────
    def train_on_experience(self, batch_size=32, num_epochs=10):
        """Train MLP on accumulated experiences with known rewards."""
        ready = [e for e in self.experience_buffer if e['reward'] is not None]
        if not ready:
            print("   ℹ Not enough rewarded experience to train MLP yet")
            return

        print(f"\n   {'='*66}")
        print(f"   TRAINING MLP ON {len(ready)} EXPERIENCES")
        print(f"   {'='*66}")

        all_scores, all_rewards = [], []
        for exp in ready:
            for idx in exp['selected_indices']:
                all_scores.append([exp['pool_scores'][m][idx] for m in self.selected_methods])
                all_rewards.append(exp['reward'])

        scores_t  = torch.tensor(all_scores,  dtype=torch.float32, device=self.device)
        rewards_t = torch.tensor(all_rewards, dtype=torch.float32, device=self.device)
        if len(rewards_t) > 1:
            rewards_t = (rewards_t - rewards_t.mean()) / (rewards_t.std() + 1e-8)

        dataset = torch.utils.data.TensorDataset(scores_t, rewards_t)
        loader  = torch.utils.data.DataLoader(
            dataset, batch_size=max(1, min(batch_size, len(scores_t))), shuffle=True)

        self.mlp.train()
        for epoch in range(num_epochs):
            epoch_loss, nb = 0.0, 0
            for sb, rb in loader:
                self.optimizer.zero_grad()
                pw       = self.mlp(sb)
                hs       = (pw * sb).sum(dim=1)
                loss     = F.mse_loss(hs, torch.sigmoid(rb), reduction='none')
                w_loss   = (loss * torch.abs(rb)).mean()
                entropy  = -(pw * torch.log(pw + 1e-8)).sum(dim=1).mean()
                total    = w_loss - 0.01 * entropy
                total.backward()
                torch.nn.utils.clip_grad_norm_(self.mlp.parameters(), 1.0)
                self.optimizer.step()
                epoch_loss += total.item(); nb += 1
            if (epoch + 1) % 5 == 0:
                print(f"   Epoch {epoch+1}/{num_epochs}: Loss = {epoch_loss/max(nb,1):.4f}")

        print(f"   ✓ MLP training complete!")
        print(f"   {'='*66}\n")

    # ── update after evaluation ────────────────────────────────────────────
    def update_weights(self, current_val_miou, iteration):
        """Called after validation each iteration. Fills reward, retrains MLP."""
        if iteration > 1 and len(self.experience_buffer) > 0:
            improvement = current_val_miou - self.previous_val_miou
            reward      = improvement * 10.0
            if current_val_miou > self.best_val_miou:
                reward             += 1.0
                self.best_val_miou  = current_val_miou
                self.best_mlp_state = {k: v.cpu().clone()
                                       for k, v in self.mlp.state_dict().items()}
                print(f"   ✓ New best validation mIoU: {self.best_val_miou:.4f}")

            self.experience_buffer[-1]['reward'] = reward

            print(f"\n   {'='*66}")
            print(f"   MLP META-LEARNER UPDATE (Iteration {iteration})")
            print(f"   {'='*66}")
            print(f"   Current mIoU  : {current_val_miou:.4f}")
            print(f"   Previous mIoU : {self.previous_val_miou:.4f}")
            print(f"   Improvement   : {improvement:+.4f}")
            print(f"   Reward        : {reward:.4f}")
            print(f"   Buffer size   : {len(self.experience_buffer)}")
            print(f"   Best mIoU     : {self.best_val_miou:.4f}")
            print(f"   {'='*66}\n")

            if iteration >= 2:
                n_ready = len([e for e in self.experience_buffer if e['reward'] is not None])
                self.train_on_experience(
                    batch_size=min(32, max(1, n_ready * 10)),
                    num_epochs=10)
                self.scheduler.step(current_val_miou)

        self.previous_val_miou = current_val_miou

        with torch.no_grad():
            probe   = torch.rand(100, self.num_methods, device=self.device)
            avg_w   = self.mlp(probe).mean(dim=0).cpu().numpy()
        self.current_weights = avg_w.tolist()

        m2w = dict(zip(self.selected_methods, avg_w))
        self.history.append({
            'iteration':            iteration,
            'val_miou':             current_val_miou,
            'best_val_miou':        self.best_val_miou,
            'experience_buffer_size': len(self.experience_buffer),
            'avg_weight_entropy':   float(m2w.get('entropy',          0.0)),
            'avg_weight_unet':      float(m2w.get('unet_diversity',   0.0)),
            'avg_weight_dino':      float(m2w.get('dinov2_diversity', 0.0)),
        })
        return tuple(avg_w)

    def get_weights(self):
        return tuple(self.current_weights)

    def save_history(self, path):
        if self.history:
            pd.DataFrame(self.history).to_csv(path, index=False)
            print(f"✓ MLP weight history saved: {path}")

    def get_state(self):
        """Serialisable state dict for checkpointing."""
        return {
            'mlp_state_dict':    self.mlp.state_dict(),
            'optimizer_state':   self.optimizer.state_dict(),
            'experience_buffer': list(self.experience_buffer),
            'best_val_miou':     self.best_val_miou,
            'best_mlp_state':    self.best_mlp_state,
            'current_weights':   self.current_weights,
            'previous_val_miou': self.previous_val_miou,
            'history':           self.history,
        }

    def load_state(self, state):
        """Restore from checkpoint state dict."""
        self.mlp.load_state_dict(state['mlp_state_dict'])
        self.optimizer.load_state_dict(state['optimizer_state'])
        self.experience_buffer  = deque(state['experience_buffer'], maxlen=100)
        self.best_val_miou      = state['best_val_miou']
        self.best_mlp_state     = state['best_mlp_state']
        self.current_weights    = state['current_weights']
        self.previous_val_miou  = state['previous_val_miou']
        self.history            = state['history']
        print(f"   ✓ MLP state restored (best_val_miou={self.best_val_miou:.4f})")


# =============================================================================
#  BLOCK 5: Visualizations
# =============================================================================


# =============================================================================

CLASS_NAMES = {
    0: "Ignore",
    1: "Background",
    2: "Building",
    3: "Road",
    4: "Water",
    5: "Barren",
    6: "Forest",
    7: "Agriculture"
}

COLOR_MAP = {
    0: (0, 0, 0),           # Ignore / NoData (Black)
    1: (255, 255, 255),     # Background (White)
    2: (255, 0, 0),         # Building (Red)
    3: (255, 255, 0),       # Road (Yellow)
    4: (0, 0, 255),         # Water (Blue)
    5: (159, 129, 183),     # Barren (Purple)
    6: (0, 255, 0),         # Forest (Green)
    7: (255, 195, 128),     # Agriculture (Orange)
}

def apply_color_map(mask, color_map):
    h, w = mask.shape
    colored_mask = np.zeros((h, w, 3), dtype=np.uint8)
    for class_id, color in color_map.items():
        colored_mask[mask == class_id] = color
    return colored_mask

def save_predicted_test_masks(model, test_loader, device, output_dir, color_map):
    """
    Inference-only pass over the test set (no ground-truth masks available).
    For each test image, predicts the segmentation mask, applies COLOR_MAP,
    and saves it to `output_dir` using the SAME filename as the input image.
    Runs once, after all active-learning iterations are complete.
    """
    os.makedirs(output_dir, exist_ok=True)
    base_model = model.module if isinstance(model, nn.DataParallel) else model
    base_model.eval()

    saved = 0
    with torch.no_grad():
        for batch in test_loader:
            if len(batch) == 3:
                images, _, filenames = batch
            else:
                images, _ = batch
                filenames = [f"image_{saved + i}.png" for i in range(images.size(0))]

            images      = images.to(device)
            outputs     = base_model(images)
            predictions = torch.argmax(outputs, dim=1).cpu().numpy()   # (B, H, W)

            for i in range(predictions.shape[0]):
                fname = filenames[i] if isinstance(filenames[i], str) else f"image_{saved}.png"
                pred_color = apply_color_map(predictions[i], color_map)      # (H, W, 3) RGB uint8
                pred_bgr   = cv2.cvtColor(pred_color, cv2.COLOR_RGB2BGR)     # cv2.imwrite expects BGR
                cv2.imwrite(os.path.join(output_dir, fname), pred_bgr)
                saved += 1

    print(f"✓ Saved {saved} predicted masks to {output_dir}")



def plot_training_results(metrics_history, save_dir):
    """Plot training metrics — validation mIoU and training-set growth only
    (no test metrics: the test set has no ground-truth labels)."""
    if len(metrics_history) == 0:
        return
    df = pd.DataFrame(metrics_history)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    if 'val_miou' in df.columns and not df['val_miou'].isna().all():
        axes[0].plot(df['iteration'], df['val_miou'], 'g-s', linewidth=2, markersize=6)
        axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('Validation mIoU')
        axes[0].set_title('Validation mIoU over Iterations', fontsize=14, fontweight='bold')
        axes[0].grid(True, alpha=0.3)
    else:
        axes[0].text(0.5, 0.5, 'No Validation Data', ha='center', va='center', fontsize=14)
        axes[0].set_title('Validation mIoU', fontsize=14, fontweight='bold')

    axes[1].plot(df['iteration'], df['train_size'], 'purple', linewidth=2, marker='s', markersize=6)
    axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('Training Set Size')
    axes[1].set_title('Training Set Growth', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path = os.path.join(save_dir, 'training_metrics.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Training metrics plot saved: {plot_path}")


# =============================================================================
#  MAIN BLOCK: Dataset helpers, checkpoint, sampling, active learning loop
# =============================================================================

class_names = ["Ignore", "Background", "Building", "Road", "Water",
               "Barren", "Forest", "Agriculture"]
num_classes = len(class_names)


class SegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir):
        self.image_dir   = image_dir
        self.mask_dir    = mask_dir
        self.image_files = sorted([f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png', '.tif'))])
        self.mask_files  = sorted([f for f in os.listdir(mask_dir)  if f.endswith(('.jpg', '.png', '.tif'))])

    def __len__(self): return len(self.image_files)

    def __getitem__(self, idx):
        image = cv2.imread(os.path.join(self.image_dir, self.image_files[idx]))
        mask  = cv2.imread(os.path.join(self.mask_dir,  self.mask_files[idx]), cv2.IMREAD_GRAYSCALE)
        if image is None or mask is None:
            raise ValueError(f"Failed to load index {idx}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)   # cv2 loads BGR → convert to RGB
        image = cv2.resize(image, (512, 512))
        mask  = cv2.resize(mask,  (512, 512), interpolation=cv2.INTER_NEAREST)
        image = torch.tensor(image).permute(2, 0, 1).float() / 255.0
        mask  = torch.clamp(torch.tensor(mask, dtype=torch.long), 0, num_classes - 1)
        return image, mask


class ValidationDataset(Dataset):
    def __init__(self, image_dir, label_dir=None):
        self.image_dir   = image_dir
        self.label_dir   = label_dir
        self.image_files = sorted([f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png', '.tif'))])
        self.has_labels  = label_dir is not None
        if self.has_labels:
            self.label_files = sorted([f for f in os.listdir(label_dir) if f.endswith(('.jpg', '.png', '.tif'))])
        else:
            self.label_files = [None] * len(self.image_files)

    def __len__(self): return len(self.image_files)

    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.image_files[idx])
        image      = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"Failed to load: {image_path}")
        image        = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)   # cv2 loads BGR → convert to RGB
        image        = cv2.resize(image, (512, 512))
        image_tensor = torch.tensor(image).permute(2, 0, 1).float() / 255.0

        if self.has_labels and self.label_files[idx]:
            label_path = os.path.join(self.label_dir, self.label_files[idx])
            if os.path.exists(label_path):
                label = cv2.imread(label_path, cv2.IMREAD_GRAYSCALE)
                if label is not None:
                    label = cv2.resize(label, (512, 512), interpolation=cv2.INTER_NEAREST)
                    label_tensor = torch.clamp(torch.tensor(label, dtype=torch.long), 0, num_classes - 1)
                else:
                    label_tensor = torch.zeros((512, 512), dtype=torch.long)
            else:
                label_tensor = torch.zeros((512, 512), dtype=torch.long)
        else:
            label_tensor = torch.zeros((512, 512), dtype=torch.long)

        return image_tensor, label_tensor, self.image_files[idx]


def _seed_worker(worker_id: int) -> None:
    """Seed each DataLoader worker so shuffling is reproducible."""
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def create_safe_dataloader(dataset, batch_size, shuffle=False, num_workers=0, use_multi_gpu=False):
    if use_multi_gpu and torch.cuda.is_available():
        num_workers = min(num_workers * torch.cuda.device_count(), 16)
    generator = torch.Generator()
    generator.manual_seed(GLOBAL_SEED)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      num_workers=num_workers, pin_memory=torch.cuda.is_available(),
                      persistent_workers=(num_workers > 0),
                      worker_init_fn=_seed_worker,
                      generator=generator)


def verify_data_consistency(image_dir, label_dir, dataset_name="Dataset"):
    if not os.path.exists(image_dir) or not os.path.exists(label_dir):
        print(f"{dataset_name}: Directories not found"); return False
    image_files = sorted([f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png', '.tif'))])
    label_files = sorted([f for f in os.listdir(label_dir) if f.endswith(('.jpg', '.png', '.tif'))])
    print(f"{dataset_name}: {len(image_files)} images, {len(label_files)} labels")
    if len(image_files) != len(label_files):
        print(f"{dataset_name}: ⚠ Mismatch in number of images and labels"); return False
    print(f"{dataset_name}: ✓ Data consistency verified"); return True


def get_sampling_methods():
    print("\n" + "="*70)
    print("SAMPLING METHOD SELECTION")
    print("="*70)
    print("\nAvailable methods:")
    print("  1. Entropy (Uncertainty-based)")
    print("  2. UNet Diversity (Task-specific latent space)")
    print("  3. GEO-DINOv2 Diversity (Satellite-specific semantic features)")
    print("\nYou can select: e.g. '1', '1,2', or '1,2,3'")
    print("="*70)
    method_map = {'1': 'entropy', '2': 'unet_diversity', '3': 'dinov2_diversity'}
    method_names = {
        'entropy':          'Entropy (Uncertainty)',
        'unet_diversity':   'UNet Diversity (Task-specific)',
        'dinov2_diversity': 'DINOv2 Diversity (Semantic features)'
    }
    while True:
        user_input = input("\nEnter method numbers (comma-separated, e.g., '1,2,3'): ").strip()
        try:
            selected_numbers  = [x.strip() for x in user_input.split(',')]
            selected_methods  = []
            for num in selected_numbers:
                if num not in method_map:
                    print(f"⚠ Invalid: {num}. Use 1, 2, or 3."); raise ValueError
                selected_methods.append(method_map[num])
            if len(selected_methods) == 0:
                print("⚠ Select at least one method."); continue
            selected_methods = list(dict.fromkeys(selected_methods))
            print(f"\n✓ Selected methods ({len(selected_methods)}):")
            for i, m in enumerate(selected_methods, 1):
                print(f"  {i}. {method_names[m]}")
            if input("\nConfirm selection? (y/n): ").strip().lower() == 'y':
                return selected_methods
            print("Let's try again...")
        except (ValueError, KeyError):
            print(" Invalid input. Please try again.")


def save_checkpoint(iteration, model, optimizer, metrics_history,
                    checkpoint_dir, model_path, selected_methods,
                    mlp_learner=None):
    """Checkpoint including MLP meta-learner state."""
    checkpoint_path = os.path.join(checkpoint_dir, 'checkpoint.pth')
    model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    ckpt = {
        'iteration':            iteration,
        'model_state_dict':     model_state,
        'optimizer_state_dict': optimizer.state_dict(),
        'selected_methods':     selected_methods,
        'metrics_history':      metrics_history,
        'last_model_path':      model_path,
    }
    if mlp_learner is not None:
        ckpt['mlp_state'] = mlp_learner.get_state()
    torch.save(ckpt, checkpoint_path)
    print(f"✓ Checkpoint saved: {checkpoint_path}")
    return checkpoint_path


def load_checkpoint(checkpoint_dir, device):
    """Load checkpoint including MLP meta-learner state if present."""
    for fname in ['checkpoint.pth', 'checkpoint_mlp.pth']:
        checkpoint_path = os.path.join(checkpoint_dir, fname)
        if os.path.exists(checkpoint_path):
            print(f"\n{'='*70}")
            print("RESUMING FROM CHECKPOINT")
            print(f"{'='*70}")
            checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
            has_mlp = 'mlp_state' in checkpoint
            print(f"Resuming from iteration  : {checkpoint['iteration'] + 1}")
            print(f"MLP state included       : {'yes' if has_mlp else 'no (will reinitialise)'}")
            print(f"{'='*70}\n")
            return checkpoint
    return None


def save_detailed_results(results, save_dir, iteration):
    results_dir  = os.path.join(save_dir, 'results')
    os.makedirs(results_dir, exist_ok=True)
    metrics_path = os.path.join(results_dir, f'detailed_metrics_iteration_{iteration}.csv')
    pd.DataFrame(results).to_csv(metrics_path, index=False)
    print(f"✓ Detailed results saved: {metrics_path}")


def perform_triple_hybrid_sampling_with_mlp(
    segmentation_model, geodino_model, geodino_processor,
    pool_loader, train_loader,
    pool_image_dir, pool_label_dir,
    target_image_dir, target_label_dir,
    selected_samples_dir, samples_per_iteration, device, iteration,
    mlp_learner, selected_methods,
    diversity_metric='cosine'
):
    """
    Hybrid active-learning sampling using MLP-predicted per-sample weights.
    Only computes scores for selected methods.
    Returns: (moved_count, selected_indices, pool_scores)
    """
    print(f"\n{'='*70}")
    print(f"TRIPLE HYBRID SAMPLING WITH MLP (Iteration {iteration + 1})")
    print(f"{'='*70}\n")

    os.makedirs(selected_samples_dir, exist_ok=True)
    selected_images_dir = os.path.join(selected_samples_dir, 'images')
    selected_labels_dir = os.path.join(selected_samples_dir, 'labels')
    os.makedirs(selected_images_dir, exist_ok=True)
    os.makedirs(selected_labels_dir, exist_ok=True)

    seg_model = segmentation_model.module if isinstance(segmentation_model, nn.DataParallel)                 else segmentation_model
    seg_model.eval()
    if geodino_model is not None:
        geodino_model.eval()

    need_unet    = 'unet_diversity'   in selected_methods
    need_dinov2  = 'dinov2_diversity' in selected_methods
    need_entropy = 'entropy'          in selected_methods

    unet_train_embeddings   = torch.empty(0, seg_model.embedding_dim).to(device)
    geodino_train_embeddings = torch.empty(0, 768).to(device)

    # ── Step 1: Training embeddings ──────────────────────────────────────────
    if need_unet or need_dinov2:
        print("Step 1/7: Extracting training embeddings...")
        unet_train_list, dino_train_list = [], []
        with torch.no_grad():
            for images, _ in train_loader:
                images = images.to(device)
                if need_unet:
                    unet_train_list.append(seg_model(images, return_embedding=True))
                if need_dinov2:
                    dino_train_list.append(
                        get_geodinov2_embeddings(images, geodino_model, geodino_processor))
        if need_unet and unet_train_list:
            unet_train_embeddings = torch.cat(unet_train_list, dim=0)
            print(f"   \u2713 UNet train embeddings  : {unet_train_embeddings.shape}")
        if need_dinov2 and dino_train_list:
            geodino_train_embeddings = torch.cat(dino_train_list, dim=0)
            print(f"   \u2713 DINOv2 train embeddings: {geodino_train_embeddings.shape}")
    else:
        print("Step 1/7: Skipping training embeddings (not needed for selected methods)")

    # ── Step 2: Pool scores ───────────────────────────────────────────────────
    print("\nStep 2/7: Computing scores for pool samples...")
    pool_data = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(pool_loader):
            if len(batch) == 3:
                images, _, filenames = batch
            else:
                images, _ = batch
                filenames = [f"image_{batch_idx}_{i}" for i in range(len(images))]
            images = images.to(device)

            seg_outputs      = None
            unet_embeddings  = None
            dino_embeddings  = None

            if need_entropy or need_unet:
                if need_unet:
                    seg_outputs, unet_embeddings = seg_model(images, return_both=True)
                else:
                    seg_outputs = seg_model(images)
            if need_dinov2:
                dino_embeddings = get_geodinov2_embeddings(images, geodino_model, geodino_processor)

            for i in range(images.size(0)):
                fname = filenames[i] if isinstance(filenames, (list, tuple)) else filenames
                sample = {'filename': fname}

                if 'entropy' in selected_methods:
                    probs_np = F.softmax(seg_outputs, dim=1)[i].cpu().numpy().transpose(1, 2, 0)
                    sample['entropy_raw'] = float(-np.sum(
                        probs_np * np.log(np.clip(probs_np, 1e-10, 1.0)), axis=-1).mean())

                if 'unet_diversity' in selected_methods:
                    sample['unet_diversity_raw'] = (
                        compute_diversity_score(unet_embeddings[i],
                                               unet_train_embeddings, metric=diversity_metric)
                        if len(unet_train_embeddings) > 0 else 1.0)

                if 'dinov2_diversity' in selected_methods:
                    sample['dinov2_diversity_raw'] = (
                        compute_diversity_score(dino_embeddings[i],
                                               geodino_train_embeddings, metric=diversity_metric)
                        if len(geodino_train_embeddings) > 0 else 1.0)

                pool_data.append(sample)

    print(f"   \u2713 Computed scores for {len(pool_data)} pool samples")
    if not pool_data:
        return 0, [], {}

    # ── Step 3: Normalise selected scores ────────────────────────────────────
    print("\nStep 3/7: Normalizing scores...")

    def _normalize(arr):
        arr = np.array(arr, dtype=np.float32)
        mn, mx = arr.min(), arr.max()
        return (arr - mn) / (mx - mn + 1e-10) if mx - mn > 1e-10 else np.full_like(arr, 0.5)

    pool_scores = {}
    for method in selected_methods:
        raw  = np.array([s[f'{method}_raw'] for s in pool_data])
        norm = _normalize(raw)
        pool_scores[method] = norm
        for i, s in enumerate(pool_data):
            s[f'{method}_normalized'] = float(norm[i])
        print(f"   \u2713 {method} normalized")

    # ── Step 4: MLP hybrid scoring ────────────────────────────────────────────
    print("\nStep 4/7: Computing hybrid scores using MLP...")
    mlp_pred_path = os.path.join(selected_samples_dir, 'mlp_predictions.csv')
    hybrid_scores, predicted_weights, _ = mlp_learner.compute_hybrid_scores_with_mlp(
        pool_scores, save_path=mlp_pred_path)

    for i, s in enumerate(pool_data):
        s['hybrid_score'] = float(hybrid_scores[i])
        for j, m in enumerate(selected_methods):
            s[f'weight_{m}'] = float(predicted_weights[i, j])

    avg_w_str = ', '.join([f"{m[:3]}={predicted_weights[:, j].mean():.3f}"
                           for j, m in enumerate(selected_methods)])
    print(f"   \u2713 Per-sample MLP weights predicted")
    print(f"   \u2713 Average weights: {avg_w_str}")

    # ── Step 5: Select top samples ────────────────────────────────────────────
    print(f"\nStep 5/7: Selecting top {samples_per_iteration} samples...")
    pool_data.sort(key=lambda x: x['hybrid_score'], reverse=True)
    selected_samples = pool_data[:samples_per_iteration]
    selected_set     = set(id(s) for s in selected_samples)
    selected_indices = [i for i, s in enumerate(pool_data) if id(s) in selected_set]
    print(f"   \u2713 Selected {len(selected_samples)} samples")

    # ── Step 6: Move files ────────────────────────────────────────────────────
    print(f"\nStep 6/7: Moving samples to training set...")
    moved_count  = 0
    selection_log = []

    for sample in selected_samples:
        fname = sample['filename']
        src_img = os.path.join(pool_image_dir,    fname)
        src_lbl = os.path.join(pool_label_dir,    fname)
        dst_img = os.path.join(target_image_dir,  fname)
        dst_lbl = os.path.join(target_label_dir,  fname)
        sel_img = os.path.join(selected_images_dir, fname)
        sel_lbl = os.path.join(selected_labels_dir, fname)

        if not os.path.exists(src_img) or not os.path.exists(src_lbl):
            # Try alternate extensions for label
            base = os.path.splitext(fname)[0]
            src_lbl = None
            for ext in ('.png', '.jpg', '.tif'):
                cand = os.path.join(pool_label_dir, base + ext)
                if os.path.exists(cand):
                    src_lbl = cand; break
            if src_lbl is None:
                print(f"   \u26a0 No label for {fname} — skipping")
                continue

        try:
            shutil.copy2(src_img, sel_img)
            shutil.copy2(src_lbl, sel_lbl)
            shutil.move(src_img,  dst_img)
            shutil.move(src_lbl,  dst_lbl)
            moved_count += 1

            log_entry = {
                'filename': fname, 'iteration': iteration + 1,
                'hybrid_score': sample['hybrid_score'], 'method': 'MLP_Triple_Hybrid'
            }
            for m in selected_methods:
                log_entry[f'{m}_normalized'] = sample.get(f'{m}_normalized', 0.0)
                log_entry[f'weight_{m}']     = sample.get(f'weight_{m}',     0.0)
            selection_log.append(log_entry)
        except Exception as e:
            print(f"   \u26a0 Failed to move {fname}: {e}")

    if selection_log:
        log_path = os.path.join(selected_samples_dir, f'selection_log_iter_{iteration + 1}.csv')
        pd.DataFrame(selection_log).to_csv(log_path, index=False)
        print(f"\n   \u2713 Selection log saved: {log_path}")

    # ── Step 7: Store experience for future MLP training ─────────────────────
    print(f"\nStep 7/7: Storing experience for future MLP training...")
    mlp_learner.store_experience(pool_scores, selected_indices)
    print(f"   \u2713 Experience stored (reward computed next iteration)")
    print(f"   \u2713 Buffer size: {len(mlp_learner.experience_buffer)}")

    print(f"\n{'='*70}")
    print(f"SELECTION SUMMARY")
    print(f"{'='*70}")
    print(f"Successfully moved : {moved_count}")
    print(f"Remaining in pool  : {len(os.listdir(pool_image_dir))}")
    print(f"Total in training  : {len(os.listdir(target_image_dir))}")
    print(f"{'='*70}\n")

    return moved_count, selected_indices, pool_scores


# =============================================================================
#  MAIN
# =============================================================================

def main():
    set_global_seed(GLOBAL_SEED)   # ← must be first — fixes ALL randomness

    print("\n" + "="*70)
    print("ACTIVE LEARNING — MLP META-LEARNER WEIGHTS")
    print("TinyWeightMLP predicts per-sample weights from method scores")
    print("Trained on validation mIoU improvement as reward signal")
    print("="*70 + "\n")

    # ── GPU selection ──────────────────────────────────────────────────────────
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available. This script requires at least one GPU.")
    num_gpus = torch.cuda.device_count()
    print(f"Available GPUs: {num_gpus}")
    for i in range(num_gpus): print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

    while True:
        try:
            gpu_choice = int(input("\nSelect GPU mode  (1 = Single GPU,  2 = Multi-GPU): ").strip())
            if gpu_choice == 1:
                device = torch.device("cuda:0"); use_multi_gpu = False
                print(f"\n✓ Single GPU selected: {torch.cuda.get_device_name(0)}"); break
            elif gpu_choice == 2:
                if num_gpus < 2:
                    print(f"  ⚠ Only {num_gpus} GPU available. Enter 1."); continue
                device = torch.device("cuda:0"); use_multi_gpu = True
                print(f"\n✓ Multi-GPU selected: {num_gpus} GPUs will be used"); break
            else:
                print("   Please enter 1 or 2.")
        except ValueError:
            print("   Invalid input.")
    print()

    # ── Method & DINOv2 config ─────────────────────────────────────────────────
    selected_methods = get_sampling_methods()
    dino_config      = select_dinov2_config()

    # MLP meta-learner is initialised after checkpoint is loaded below
    mlp_learner = None

    # ── Paths ──────────────────────────────────────────────────────────────────
    BASE_DIR       = r"E:\Loveda data\loveda_8class"
    RESULTS_DIR    = os.path.join(BASE_DIR, "AMEFAL_8_class_OUTPUTS")
    DINO_CACHE_DIR = os.path.join(RESULTS_DIR, 'dinov2_cache')
    os.makedirs(DINO_CACHE_DIR, exist_ok=True)

    TRAIN_IMAGES = os.path.join(BASE_DIR, "train_data")
    TRAIN_LABELS = os.path.join(BASE_DIR, "train_labels")
    POOL_IMAGES  = os.path.join(BASE_DIR, "Unlabeled_data")
    POOL_LABELS  = os.path.join(BASE_DIR, "Validation_labels")
    VAL_IMAGES   = os.path.join(BASE_DIR, "val_img")
    VAL_LABELS   = os.path.join(BASE_DIR, "val_lab")
    TEST_IMAGES  = os.path.join(BASE_DIR, "test_img")
    TEST_LABELS  = os.path.join(BASE_DIR, "test_lab")

    for d in [RESULTS_DIR, TRAIN_IMAGES, TRAIN_LABELS, POOL_IMAGES,
              POOL_LABELS, VAL_IMAGES, VAL_LABELS, TEST_IMAGES, TEST_LABELS]:
        os.makedirs(d, exist_ok=True)

    # ── Checkpoint ────────────────────────────────────────────────────────────
    checkpoint = load_checkpoint(RESULTS_DIR, device)

    if checkpoint is not None:
        start_iteration     = checkpoint['iteration'] + 1
        metrics_history     = checkpoint['metrics_history']
        previous_model_path = checkpoint['last_model_path']
        if 'selected_methods' in checkpoint:
            selected_methods = checkpoint['selected_methods']
            print(f"✓ Loaded selected methods: {selected_methods}")

        if not os.path.exists(previous_model_path):
            print(f"\n⚠ Previous model not found: {previous_model_path}")
            print("Searching for the latest available model...")
            found_model = None
            for iter_num in range(checkpoint['iteration'], 0, -1):
                potential = os.path.join(RESULTS_DIR, f"iteration_{iter_num}", f"model_iteration_{iter_num}.pth")
                if os.path.exists(potential):
                    found_model         = potential
                    start_iteration     = iter_num + 1
                    metrics_history     = checkpoint['metrics_history'][:iter_num]
                    previous_model_path = found_model
                    print(f"✓ Found model from iteration {iter_num}")
                    break
            if found_model is None:
                print("No valid model found. Starting from scratch...")
                checkpoint          = None
                start_iteration     = 0
                metrics_history     = []
                previous_model_path = None

        if checkpoint is not None:
            print(f"\n✓ Resuming from iteration {start_iteration}")
            print(f"✓ Loaded {len(metrics_history)} previous metrics")
    else:
        start_iteration     = 0
        metrics_history     = []
        previous_model_path = None
        print("Starting new training from scratch\n")

    # ── Initialise MLP meta-learner (restore from checkpoint if available) ─
    mlp_learner = MLPMetaWeightLearner(
        selected_methods=selected_methods,
        hidden_dim=32,
        learning_rate=0.001,
        device=device)
    if checkpoint is not None and 'mlp_state' in checkpoint:
        mlp_learner.load_state(checkpoint['mlp_state'])
        print(f"\u2713 MLP meta-learner restored from checkpoint")
    else:
        print(f"\u2713 MLP meta-learner initialised fresh (equal weights)")
    print(f"  Methods : {selected_methods}")
    print(f"  Initial weights: {[round(w,4) for w in mlp_learner.get_weights()]}\n")

    # ── Data verification ─────────────────────────────────────────────────────
    print("Data Verification:")
    print("-" * 70)
    train_ok = verify_data_consistency(TRAIN_IMAGES, TRAIN_LABELS, "Training")
    pool_ok  = verify_data_consistency(POOL_IMAGES,  POOL_LABELS,  "Pool")
    val_ok   = verify_data_consistency(VAL_IMAGES,   VAL_LABELS,   "Validation")
    # No ground-truth masks for the test set — just check images exist.
    test_ok  = os.path.exists(TEST_IMAGES) and len(os.listdir(TEST_IMAGES)) > 0
    print(f"Test: {len(os.listdir(TEST_IMAGES)) if os.path.exists(TEST_IMAGES) else 0} images "
          f"(no labels — inference only)")
    print()

    if not all([train_ok, pool_ok, test_ok]):
        print(" Data consistency issues found!"); return

    # ── Configuration ─────────────────────────────────────────────────────────

    MAX_ITERATIONS          = 1
    SAMPLES_PER_ITERATION   = 50
    DINO_FINETUNE_INTERVAL  = 4
    EPOCHS_FIRST_ITERATION  = 100
    EPOCHS_SUBSEQUENT_MAX   = 50
    EARLY_STOPPING_PATIENCE = 10
    BATCH_SIZE              = 4
    LEARNING_RATE           = 0.0001
    NUM_WORKERS             = 0
    DIVERSITY_METRIC        = 'cosine'
    DINOV2_MODEL            = 'facebook/dinov2-base'
    EMBEDDING_DIM           = 256

    print("Configuration:")
    print("-" * 70)
    print(f"  UNet Embedding Dim       : {EMBEDDING_DIM}")
    print(f"  Diversity Metric         : {DIVERSITY_METRIC}")
    print(f"  DINOv2 Model             : {DINOV2_MODEL}")
    print(f"  LoRA Rank (fixed)        : {LORA_RANK}  |  Alpha: {LORA_ALPHA}  |  Scale: {LORA_ALPHA/LORA_RANK:.1f}")
    print(f"  MLP Meta-Learner         : TinyWeightMLP (hidden_dim=32, lr=0.001)")
    print(f"  Max Iterations           : {MAX_ITERATIONS}")
    print(f"  Samples / Iteration      : {SAMPLES_PER_ITERATION}")
    print(f"  Epochs (first iter)      : {EPOCHS_FIRST_ITERATION}  [no early stopping]")
    print(f"  Epochs (subsequent max)  : {EPOCHS_SUBSEQUENT_MAX}  [early stopping patience={EARLY_STOPPING_PATIENCE}]")
    print(f"  DINOv2 re-finetune every : {DINO_FINETUNE_INTERVAL} iterations")
    if start_iteration > 0: print(f"\n  RESUMING from iteration {start_iteration}")
    print()

    # ── Load DINOv2 ───────────────────────────────────────────────────────────
    print("Loading DINOv2 encoder...")
    geodino_model, geodino_processor, geodino_embed_dim = load_geodinov2_encoder(
        device, model_name=DINOV2_MODEL, cache_dir=DINO_CACHE_DIR)
    print(f"✓ DINOv2 embedding dimension: {geodino_embed_dim}\n")

    finetuned_dino_path = os.path.join(RESULTS_DIR, 'finetuned_dinov2.pth')

    if start_iteration == 0:
        print("INITIAL DINOV2 FINE-TUNING (before iteration 0)")
        print("="*70)
        initial_train_dataset = SegmentationDataset(TRAIN_IMAGES, TRAIN_LABELS)
        if len(initial_train_dataset) > 0:
            initial_train_loader = create_safe_dataloader(
                initial_train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                num_workers=NUM_WORKERS, use_multi_gpu=use_multi_gpu)
            geodino_model, geodino_processor = run_dinov2_finetuning(
                config=dino_config, train_loader=initial_train_loader,
                device=device, num_classes=num_classes, num_epochs=15, lr=1e-4,
                model_name=DINOV2_MODEL, train_image_dir=TRAIN_IMAGES,
                pool_image_dir=POOL_IMAGES, batch_size=BATCH_SIZE,
                cache_dir=DINO_CACHE_DIR, existing_dino_model=None,
                use_gradient_checkpointing=False)
            save_finetuned_dinov2(geodino_model, geodino_processor, finetuned_dino_path)
        else:
            print("⚠ No training data — using pretrained DINOv2 without fine-tuning")
    else:
        print(" Resuming — loading existing fine-tuned DINOv2...")
        load_finetuned_dinov2(geodino_model, finetuned_dino_path, device)
        print()

    # ── Validation & test loaders ──────────────────────────────────────────────
    if val_ok and os.path.exists(VAL_IMAGES) and len(os.listdir(VAL_IMAGES)) > 0:
        val_dataset = ValidationDataset(VAL_IMAGES, VAL_LABELS)
        val_loader  = create_safe_dataloader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                                             num_workers=NUM_WORKERS, use_multi_gpu=use_multi_gpu)
        use_validation = True
        print(f"Validation samples: {len(val_dataset)}")
    else:
        val_loader     = None
        use_validation = False
        print(" No validation set — performance tracking will be limited!")

    test_dataset = ValidationDataset(TEST_IMAGES, None)   # no test labels — inference only
    test_loader  = create_safe_dataloader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                                          num_workers=NUM_WORKERS, use_multi_gpu=use_multi_gpu)
    print(f"Test samples: {len(test_dataset)}\n")

    criterion = nn.CrossEntropyLoss(ignore_index=0)  # class 0 = Ignore/NoData — excluded from loss

    # ── Active Learning Loop ───────────────────────────────────────────────────
    try:
        for iteration in range(start_iteration, MAX_ITERATIONS):
            current_iteration = iteration

            print("\n" + "="*70)
            print(f"ITERATION {current_iteration}/{MAX_ITERATIONS}")
            print("="*70 + "\n")

            iteration_dir    = os.path.join(RESULTS_DIR, f"iteration_{current_iteration}")
            os.makedirs(iteration_dir, exist_ok=True)

            pool_images_list = [f for f in os.listdir(POOL_IMAGES) if f.endswith(('.jpg', '.png', '.tif'))]
            if len(pool_images_list) == 0:
                print(" Pool is empty!"); break

            print(f"Dataset sizes:")
            print(f"  Training : {len(os.listdir(TRAIN_IMAGES))}")
            print(f"  Pool     : {len(pool_images_list)}")
            print(f"  Test     : {len(test_dataset)}\n")

            # ── STEP 1: Train UNet ─────────────────────────────────────────────
            print("STEP 1: Training UNet with Latent Space")
            print("-" * 70)
            train_dataset = SegmentationDataset(TRAIN_IMAGES, TRAIN_LABELS)
            train_loader  = create_safe_dataloader(
                train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                num_workers=NUM_WORKERS, use_multi_gpu=use_multi_gpu)

            model     = UNetWithLatentSpace(num_classes=num_classes, embedding_dim=EMBEDDING_DIM)
            optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
            is_first_iteration = (iteration == 0)
            max_epochs = EPOCHS_FIRST_ITERATION if is_first_iteration else EPOCHS_SUBSEQUENT_MAX

            model, train_losses, epochs_completed, val_losses, val_mious = train_model_with_warm_start(
                train_loader, model, criterion, optimizer, max_epochs, device,
                use_multi_gpu=use_multi_gpu, previous_model_path=previous_model_path,
                val_loader=val_loader if use_validation else None,
                early_stopping_patience=EARLY_STOPPING_PATIENCE,
                is_first_iteration=is_first_iteration)

            # ── STEP 2: Evaluation ─────────────────────────────────────────────
            # No test-set ground truth is available, so only validation is
            # evaluated per iteration. Test predictions are generated once,
            # after ALL iterations finish (see end of the active-learning loop).
            print("\nSTEP 2: Evaluation")
            print("-" * 70)

            if use_validation:
                val_acc, val_loss_ev, val_miou, val_class_ious, val_class_accs = validate_model(
                    model, val_loader, criterion, device, num_classes=num_classes)
                print(f"Validation — Acc: {val_acc:.4f} | Loss: {val_loss_ev:.4f} | mIoU: {val_miou:.4f}")
            else:
                val_miou = 0.0
                print("Validation — Skipped (no validation set)")

            # ── RE-FINE-TUNE DINOV2 ────────────────────────────────────────────
            if current_iteration > 0 and current_iteration % DINO_FINETUNE_INTERVAL == 0:
                print("\n" + "="*70)
                print(f"RE-FINE-TUNING DINOV2 (Iteration {current_iteration})")
                print("="*70)
                current_train_dataset = SegmentationDataset(TRAIN_IMAGES, TRAIN_LABELS)
                current_train_loader  = create_safe_dataloader(
                    current_train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=NUM_WORKERS, use_multi_gpu=use_multi_gpu)
                num_train_now = len(os.listdir(TRAIN_IMAGES))
                # Move old model to CPU and free GPU memory before loading new one
                geodino_model.cpu()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                geodino_model, geodino_processor = run_dinov2_finetuning(
                    config=dino_config, train_loader=current_train_loader,
                    device=device, num_classes=num_classes, num_epochs=15, lr=1e-4,
                    model_name=DINOV2_MODEL, train_image_dir=TRAIN_IMAGES,
                    pool_image_dir=POOL_IMAGES, batch_size=BATCH_SIZE,
                    cache_dir=DINO_CACHE_DIR, existing_dino_model=geodino_model,
                    use_gradient_checkpointing=(num_train_now > 500))
                save_finetuned_dinov2(geodino_model, geodino_processor, finetuned_dino_path)
                print(f"\n✓ DINOv2 re-fine-tuned and saved!")
                print("="*70 + "\n")

            # ── STEP 3: MLP weight update ──────────────────────────────────────
            print("\nSTEP 3: MLP Meta-Learner Weight Update")
            print("-" * 70)
            current_avg_weights = mlp_learner.update_weights(
                current_val_miou=val_miou,
                iteration=current_iteration)
            method_display = {'entropy': 'Entropy', 'unet_diversity': 'UNet Div', 'dinov2_diversity': 'DINOv2 Div'}
            print(f"   MLP avg weights this iteration:")
            for m, w in zip(selected_methods, current_avg_weights):
                print(f"     {method_display[m]}: {w:.4f}  [MLP-PREDICTED]")

            # ── STEP 4: Save results ───────────────────────────────────────────
            results = {
                'iteration':                       current_iteration,
                'train_size':                      len(train_dataset),
                'pool_size':                       len(pool_images_list),
                'val_miou':                        val_miou if use_validation else np.nan,
                'epochs_trained':                  epochs_completed,
                'avg_entropy_weight':              float(dict(zip(selected_methods, current_avg_weights)).get('entropy',          0.0)),
                'avg_unet_diversity_weight':       float(dict(zip(selected_methods, current_avg_weights)).get('unet_diversity',   0.0)),
                'avg_dinov2_diversity_weight':     float(dict(zip(selected_methods, current_avg_weights)).get('dinov2_diversity', 0.0)),
                'sampling_method':                 'mlp_meta_learner',
                'diversity_metric':                DIVERSITY_METRIC,
                'lora_rank':                       LORA_RANK,
            }
            if use_validation:
                for i, cls_name in enumerate(class_names):
                    results[f'val_iou_{cls_name}'] = val_class_ious[i]

            metrics_history.append(results)
            save_detailed_results([results], iteration_dir, current_iteration)

            model_path          = save_model(model, iteration_dir, current_iteration, num_classes=num_classes)
            previous_model_path = model_path

            # No per-iteration visualizations / test predictions — predicted
            # masks for the test set are generated ONCE, after the final
            # iteration (see below, after this loop ends).

            # ── STEP 5: Save checkpoint ────────────────────────────────────────
            save_checkpoint(current_iteration, model, optimizer, metrics_history,
                            RESULTS_DIR, model_path, selected_methods,
                            mlp_learner=mlp_learner)

            # ── STEP 6: Active Learning Sampling ──────────────────────────────
            if len(pool_images_list) > 0:
                print("\nSTEP 4: Active Learning Sample Selection (fixed weights)")
                print("-" * 70)

                pool_dataset = ValidationDataset(POOL_IMAGES, POOL_LABELS)
                pool_loader  = create_safe_dataloader(
                    pool_dataset, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, use_multi_gpu=use_multi_gpu)

                selected_samples_dir = os.path.join(iteration_dir, 'selected_samples')

                moved_count, selected_indices, pool_scores = perform_triple_hybrid_sampling_with_mlp(
                    segmentation_model=model,
                    geodino_model=geodino_model,
                    geodino_processor=geodino_processor,
                    pool_loader=pool_loader,
                    train_loader=train_loader,
                    pool_image_dir=POOL_IMAGES,
                    pool_label_dir=POOL_LABELS,
                    target_image_dir=TRAIN_IMAGES,
                    target_label_dir=TRAIN_LABELS,
                    selected_samples_dir=selected_samples_dir,
                    samples_per_iteration=SAMPLES_PER_ITERATION,
                    device=device,
                    iteration=current_iteration,
                    mlp_learner=mlp_learner,
                    selected_methods=selected_methods,
                    diversity_metric=DIVERSITY_METRIC)

                # Save MLP weight history after every iteration
                mlp_history_path = os.path.join(RESULTS_DIR, "mlp_weight_history.csv")
                mlp_learner.save_history(mlp_history_path)

                print(f"\n✓ Added {moved_count} samples to training set")
            else:
                print("\n⚠ No samples remaining in pool")
                break

            # ── Cleanup ────────────────────────────────────────────────────────
            # Zero gradients before deleting model to avoid retained graph refs
            try:
                model.zero_grad(set_to_none=True)
            except Exception:
                pass
            del model, optimizer, train_loader, pool_loader
            # pool_loader_viz may have been created for embedding viz
            if 'pool_loader_viz' in dir():
                try:
                    del pool_loader_viz
                except Exception:
                    pass
            plt.close('all')                        # close any leaked figures
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.synchronize()
            gc.collect()

            print(f"\n{'='*70}")
            print(f"ITERATION {current_iteration} COMPLETED")
            print(f"{'='*70}\n")

    except KeyboardInterrupt:
        print("\n\n⚠ Training interrupted by user")
        print("Checkpoint saved — you can resume later")

    except Exception as e:
        print(f"\n\n Error occurred: {str(e)}")
        import traceback; traceback.print_exc()

    finally:
        print("\n" + "="*70)
        print("TRAINING SUMMARY — MLP META-LEARNER")
        print("="*70)

        if len(metrics_history) > 0:
            summary_df   = pd.DataFrame(metrics_history)
            summary_path = os.path.join(RESULTS_DIR, 'training_summary.csv')
            summary_df.to_csv(summary_path, index=False)
            print(f"✓ Training summary saved: {summary_path}")
            print(f"\nCompleted {len(metrics_history)} iterations")
            if 'val_miou' in metrics_history[-1]:
                print(f"Final val mIoU : {metrics_history[-1]['val_miou']:.4f}")
                print(f"Best  val mIoU : {max(m['val_miou'] for m in metrics_history):.4f}")
            plot_training_results(metrics_history, RESULTS_DIR)

        # ── FINAL STEP: Predict on test set (no ground truth) and save masks ──
        # Runs once, after ALL iterations complete — loads the last saved
        # model (since `model` is deleted at the end of every iteration).
        print(f"\n{'='*70}")
        print("GENERATING TEST PREDICTIONS (final model)")
        print(f"{'='*70}")
        if previous_model_path and os.path.exists(previous_model_path):
            final_ckpt  = torch.load(previous_model_path, map_location=device, weights_only=False)
            final_model = UNetWithLatentSpace(
                num_classes=final_ckpt.get('num_classes', num_classes),
                embedding_dim=EMBEDDING_DIM).to(device)
            final_model.load_state_dict(final_ckpt['model_state_dict'])
            test_masks_dir = os.path.join(RESULTS_DIR, 'test_masks')
            save_predicted_test_masks(
                final_model, test_loader, device, test_masks_dir, COLOR_MAP)
            del final_model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        else:
            print("⚠ No saved model found — skipping test mask generation.")

        print(f"\n{'='*70}")
        print("ALL RESULTS SAVED TO:")
        print(f"  {RESULTS_DIR}")
        print(f"{'='*70}\n")


if __name__ == "__main__":
    main()


✓ Global seed set to 42 — fully reproducible run

ACTIVE LEARNING — MLP META-LEARNER WEIGHTS
TinyWeightMLP predicts per-sample weights from method scores
Trained on validation mIoU improvement as reward signal

Available GPUs: 1
  GPU 0: NVIDIA RTX 4000 Ada Generation



Select GPU mode  (1 = Single GPU,  2 = Multi-GPU):  1



✓ Single GPU selected: NVIDIA RTX 4000 Ada Generation


SAMPLING METHOD SELECTION

Available methods:
  1. Entropy (Uncertainty-based)
  2. UNet Diversity (Task-specific latent space)
  3. GEO-DINOv2 Diversity (Satellite-specific semantic features)

You can select: e.g. '1', '1,2', or '1,2,3'



Enter method numbers (comma-separated, e.g., '1,2,3'):  1,2,3



✓ Selected methods (3):
  1. Entropy (Uncertainty)
  2. UNet Diversity (Task-specific)
  3. DINOv2 Diversity (Semantic features)


...

  LoRA: 6 projections injected | trainable=36,864 params
  ✓ LoRA warm-start: loaded=12 keys, skipped=0 keys
  ✓ Gradient checkpointing ENABLED (saves ~30% memory)
  Total trainable LoRA params: 36,864
  Head params:                 527,112
Training...
----------------------------------------------------------------------
  Epoch   1/15 | Loss: 0.4482 | LR: 9.90e-05
  Epoch   3/15 | Loss: 0.2872 | LR: 9.14e-05
  Epoch   6/15 | Loss: 0.2274 | LR: 6.89e-05
  Epoch   9/15 | Loss: 0.2209 | LR: 4.11e-05
  Epoch  12/15 | Loss: 0.2184 | LR: 1.86e-05
  Epoch  15/15 | Loss: 0.2170 | LR: 1.00e-05

✓ Done! | Best loss: 0.2170

✓ Fine-tuned DINOv2 saved: E:\Loveda data\loveda_8class\AMEFAL_8_class_OUTPUTS\finetuned_dinov2.pth
  Saved 48 weight tensors (12 LoRA keys)

✓ DINOv2 re-fine-tuned and saved!


STEP 3: MLP Meta-Learner Weight Update
----------------------------------------------------------------------
   ✓ New best validation mIoU: 0.4673

   MLP META-LEARNER UPDATE (Iteration 8)
   Curre

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

  LoRA: 6 projections injected | trainable=36,864 params
  ✓ LoRA warm-start: loaded=12 keys, skipped=0 keys
  ✓ Gradient checkpointing ENABLED (saves ~30% memory)
  Total trainable LoRA params: 36,864
  Head params:                 527,112
Training...
----------------------------------------------------------------------
  Epoch   1/15 | Loss: 0.4284 | LR: 9.90e-05
  Epoch   3/15 | Loss: 0.2622 | LR: 9.14e-05
  Epoch   6/15 | Loss: 0.2293 | LR: 6.89e-05
  Epoch   9/15 | Loss: 0.2247 | LR: 4.11e-05
  Epoch  12/15 | Loss: 0.2224 | LR: 1.86e-05
  Epoch  15/15 | Loss: 0.2211 | LR: 1.00e-05

✓ Done! | Best loss: 0.2211

✓ Fine-tuned DINOv2 saved: E:\Loveda data\loveda_8class\AMEFAL_8_class_OUTPUTS\finetuned_dinov2.pth
  Saved 48 weight tensors (12 LoRA keys)

✓ DINOv2 re-fine-tuned and saved!


STEP 3: MLP Meta-Learner Weight Update
----------------------------------------------------------------------
   ✓ New best validation mIoU: 0.4919

   MLP META-LEARNER UPDATE (Iteration 12)
   Curr

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

  LoRA: 6 projections injected | trainable=36,864 params
  ✓ LoRA warm-start: loaded=12 keys, skipped=0 keys
  ✓ Gradient checkpointing ENABLED (saves ~30% memory)
  Total trainable LoRA params: 36,864
  Head params:                 527,112
Training...
----------------------------------------------------------------------
  Epoch   1/15 | Loss: 0.4113 | LR: 9.90e-05
  Epoch   3/15 | Loss: 0.2531 | LR: 9.14e-05
  Epoch   6/15 | Loss: 0.2319 | LR: 6.89e-05
  Epoch   9/15 | Loss: 0.2278 | LR: 4.11e-05
  Epoch  12/15 | Loss: 0.2247 | LR: 1.86e-05
  Epoch  15/15 | Loss: 0.2238 | LR: 1.00e-05

✓ Done! | Best loss: 0.2238

✓ Fine-tuned DINOv2 saved: E:\Loveda data\loveda_8class\AMEFAL_8_class_OUTPUTS\finetuned_dinov2.pth
  Saved 48 weight tensors (12 LoRA keys)

✓ DINOv2 re-fine-tuned and saved!


STEP 3: MLP Meta-Learner Weight Update
----------------------------------------------------------------------
   ✓ New best validation mIoU: 0.5032

   MLP META-LEARNER UPDATE (Iteration 16)
   Curr

#### Fully Supervised  model